In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('../dataGenerated/gurgaon_properties_missing_value_imputation.csv')

In [4]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3573 entries, 0 to 3572
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3573 non-null   object 
 1   society          3573 non-null   object 
 2   sector           3573 non-null   object 
 3   price            3573 non-null   float64
 4   price_per_sqft   3573 non-null   float64
 5   areaWithType     3573 non-null   object 
 6   bedRoom          3573 non-null   int64  
 7   bathroom         3573 non-null   int64  
 8   balcony          3573 non-null   object 
 9   floorNum         3573 non-null   float64
 10  facing           3573 non-null   object 
 11  built_up_area    3573 non-null   float64
 12  study room       3573 non-null   int64  
 13  servant room     3573 non-null   int64  
 14  store room       3573 non-null   int64  
 15  pooja room       3573 non-null   int64  
 16  others           3573 non-null   int64  
 17  age_category  

In [6]:
import requests
import time

def get_gurugram_sector_coordinates(sectors):
    """
    Get latitude and longitude for Gurugram sectors.
    Returns a dictionary: {sector: (latitude, longitude)}
    """

    results = {}

    headers = {
        "User-Agent": "gurugram-sector-geocoder/1.0"
    }

    for sector in sectors:
        query = f"{sector}, Gurugram, Haryana, India"

        url = "https://nominatim.openstreetmap.org/search"
        params = {
            "q": query,
            "format": "json",
            "limit": 1
        }

        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=10
            )
            data = response.json()

            if data:
                results[sector] = (
                    float(data[0]["lat"]),
                    float(data[0]["lon"])
                )
            else:
                results[sector] = (None, None)

        except Exception:
            results[sector] = (None, None)

        time.sleep(1)  # respect Nominatim rate limits

    return results

In [7]:
unique_sectors = df["sector"].dropna().unique()

coordinates = get_gurugram_sector_coordinates(unique_sectors)

df["latitude"] = df["sector"].map(
    lambda x: coordinates.get(x, (None, None))[0]
)

df["longitude"] = df["sector"].map(
    lambda x: coordinates.get(x, (None, None))[1]
)

In [8]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.466257,77.014265
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.497391,77.020526
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.399939,77.045265
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.411019,77.096368
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.408905,76.915523


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3573 entries, 0 to 3572
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3573 non-null   object 
 1   society          3573 non-null   object 
 2   sector           3573 non-null   object 
 3   price            3573 non-null   float64
 4   price_per_sqft   3573 non-null   float64
 5   areaWithType     3573 non-null   object 
 6   bedRoom          3573 non-null   int64  
 7   bathroom         3573 non-null   int64  
 8   balcony          3573 non-null   object 
 9   floorNum         3573 non-null   float64
 10  facing           3573 non-null   object 
 11  built_up_area    3573 non-null   float64
 12  study room       3573 non-null   int64  
 13  servant room     3573 non-null   int64  
 14  store room       3573 non-null   int64  
 15  pooja room       3573 non-null   int64  
 16  others           3573 non-null   int64  
 17  age_category  

In [10]:
df['latitude'] = df['latitude'].fillna(28.4947)
df['longitude'] = df['longitude'].fillna(77.0226)

In [11]:
df['latitude'] = df['latitude'].round(4)
df['longitude'] = df['longitude'].round(4)

In [12]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.4663,77.0143
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.4974,77.0205
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.3999,77.0453
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.4110,77.0964
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.4089,76.9155


In [13]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

In [14]:
CYBER_CITY = (28.4946, 77.0888)

df["dist_cyber_city_km"] = df.apply(
    lambda x: haversine(
        x["latitude"], x["longitude"],
        CYBER_CITY[0], CYBER_CITY[1]
    ),
    axis=1
)

In [15]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude,dist_cyber_city_km
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.4663,77.0143,7.932372
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.4974,77.0205,6.681787
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.3999,77.0453,11.356574
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.4110,77.0964,9.325542
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.4089,76.9155,19.438662


In [16]:
df.corr(numeric_only=True)['price'].sort_values(ascending=False)

price                 1.000000
price_per_sqft        0.793641
built_up_area         0.724482
bathroom              0.610240
bedRoom               0.572662
longitude             0.403285
servant room          0.395196
pooja room            0.323536
store room            0.307230
combined_rating       0.271825
study room            0.246184
latitude              0.142623
luxury_score          0.073795
others               -0.016226
floorNum             -0.090450
dist_cyber_city_km   -0.380514
Name: price, dtype: float64

In [19]:
df['sector'].value_counts().unique()

array([152, 115, 110, 108, 103, 100,  93,  89,  87,  86,  75,  68,  67,
        62,  61,  60,  59,  58,  57,  55,  49,  46,  45,  43,  42,  41,
        40,  38,  35,  32,  31,  30,  29,  27,  26,  24,  23,  22,  21,
        20,  19,  18,  17,  16,  15,  14,  13,  11,  10,   9,   8,   7,
         6,   5,   3,   2])

In [20]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(
        radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


def distance_to_nearest_metro(lat, lon, metro_stations):
    """
    lat, lon: property coordinates
    metro_stations: list of (latitude, longitude)
    
    Returns distance to nearest metro in km.
    """

    if lat is None or lon is None:
        return None

    distances = [
        haversine(lat, lon, metro_lat, metro_lon)
        for metro_lat, metro_lon in metro_stations
    ]

    return min(distances)

In [21]:
metro_stations = [
    # Yellow Line
    (28.4990, 77.0895),  # Guru Dronacharya
    (28.4945, 77.0888),  # Sikandarpur
    (28.4827, 77.1057),  # Phase 2
    (28.4800, 77.1038),  # Phase 3
    (28.4788, 77.0925),  # Moulsari Avenue
    (28.4707, 77.0729),  # MG Road
    (28.4595, 77.0726),  # IFFCO Chowk
    (28.4508, 77.0437),  # Millennium City Centre

    # Rapid Metro
    (28.4805, 77.0965),  # DLF Phase 2
    (28.4823, 77.0940),  # Belvedere Towers
    (28.4845, 77.0915),  # Cyber City
    (28.4885, 77.0940),  # Sikandarpur
    (28.4930, 77.0970),  # Phase 2
    (28.4970, 77.1010),  # Phase 3
    (28.5010, 77.1035),  # Phase 1
    (28.5050, 77.1050),  # Sector 42-43
    (28.5065, 77.0970),  # Sector 53-54
    (28.5100, 77.0920),  # Sector 54 Chowk
    (28.5140, 77.0870),  # Sector 55-56
]

In [23]:
df["dist_nearest_metro_km"] = df.apply(
    lambda row: distance_to_nearest_metro(
        row["latitude"],
        row["longitude"],
        metro_stations
    ),
    axis=1
)

In [24]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude,dist_cyber_city_km,dist_nearest_metro_km
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.4663,77.0143,7.932372,3.351261
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.4974,77.0205,6.681787,5.656158
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.3999,77.0453,11.356574,5.661984
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.4110,77.0964,9.325542,5.873639
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.4089,76.9155,19.438662,13.373809


In [25]:
df.corr(numeric_only=True)['price'].sort_values(ascending=False)

price                    1.000000
price_per_sqft           0.793641
built_up_area            0.724482
bathroom                 0.610240
bedRoom                  0.572662
longitude                0.403285
servant room             0.395196
pooja room               0.323536
store room               0.307230
combined_rating          0.271825
study room               0.246184
latitude                 0.142623
luxury_score             0.073795
others                  -0.016226
floorNum                -0.090450
dist_nearest_metro_km   -0.360024
dist_cyber_city_km      -0.380514
Name: price, dtype: float64

In [26]:
df['combined_distance']=df['dist_cyber_city_km'] + df['dist_nearest_metro_km']

In [27]:
df.corr(numeric_only=True)['price'].sort_values(ascending=False)

price                    1.000000
price_per_sqft           0.793641
built_up_area            0.724482
bathroom                 0.610240
bedRoom                  0.572662
longitude                0.403285
servant room             0.395196
pooja room               0.323536
store room               0.307230
combined_rating          0.271825
study room               0.246184
latitude                 0.142623
luxury_score             0.073795
others                  -0.016226
floorNum                -0.090450
dist_nearest_metro_km   -0.360024
combined_distance       -0.378100
dist_cyber_city_km      -0.380514
Name: price, dtype: float64

In [32]:
df.drop(columns=['latitude','longitude'], inplace=True)

In [ ]:
df.info()

In [34]:
df.head(1)

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,dist_cyber_city_km
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.0,7.932372


In [35]:
df.drop(columns=['society','areaWithType'],inplace=True)

In [36]:
def categorize_luxury(score):
    if 0 <= score < 50:
        return "Low"
    elif 50 <= score < 90:
        return "Medium"
    elif 90 <= score <= 150:
        return "High"
    else:
        return None

In [37]:
df['luxury_category']=df['luxury_score'].apply(categorize_luxury)

In [38]:
def categorize_floor(floor):
    if 0 <= floor <= 2:
        return "Low Floor"
    elif 3 <= floor <= 10:
        return "Mid Floor"
    elif 11 <= floor <= 51:
        return "High Floor"
    else:
        return None

In [40]:
df['floor_category']=df['floorNum'].apply(categorize_floor)

In [41]:
df.head(1)

,property_type,sector,price,price_per_sqft,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,dist_cyber_city_km,luxury_category,floor_category
0,flat,sector 7,0.45,5000.0,2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.0,7.932372,Low,Mid Floor


In [42]:
df.drop(columns=['luxury_score','floorNum'],inplace=True)

In [44]:
df.drop(columns=['study room','store room','pooja room','others'],inplace=True)

In [45]:
def convert_rating(rating):
    return int(rating)

In [46]:
df['combined_rating']=df['combined_rating'].apply(convert_rating)

In [47]:
from sklearn.preprocessing import OrdinalEncoder

# Create a copy of the original data for label encoding
data_label_encoded = df.copy()

categorical_cols = df.select_dtypes(include=['object']).columns

# Apply label encoding to categorical columns
for col in categorical_cols:
    oe = OrdinalEncoder()
    data_label_encoded[col] = oe.fit_transform(data_label_encoded[[col]])
    print(oe.categories_)

# Splitting the dataset into training and testing sets
X_label = data_label_encoded.drop('price', axis=1)
y_label = data_label_encoded['price']

[array(['flat', 'house'], dtype=object)]
[array(['dwarka expressway', 'gwal pahari', 'manesar', 'sector 1',
       'sector 10', 'sector 102', 'sector 103', 'sector 104',
       'sector 105', 'sector 106', 'sector 107', 'sector 108',
       'sector 109', 'sector 11', 'sector 110', 'sector 111',
       'sector 112', 'sector 113', 'sector 12', 'sector 13', 'sector 14',
       'sector 15', 'sector 17', 'sector 2', 'sector 21', 'sector 22',
       'sector 23', 'sector 24', 'sector 25', 'sector 26', 'sector 27',
       'sector 28', 'sector 3', 'sector 3 phase 2',
       'sector 3 phase 3 extension', 'sector 30', 'sector 31',
       'sector 33', 'sector 36', 'sector 37', 'sector 38', 'sector 39',
       'sector 4', 'sector 40', 'sector 41', 'sector 43', 'sector 45',
       'sector 46', 'sector 47', 'sector 48', 'sector 49', 'sector 5',
       'sector 50', 'sector 51', 'sector 52', 'sector 53', 'sector 54',
       'sector 55', 'sector 56', 'sector 57', 'sector 58', 'sector 59',
       'sector 

In [55]:
# 1.
fi_df1 = data_label_encoded.corr()['price'].iloc[1:].to_frame().reset_index().rename(columns={'index':'feature','price':'corr_coeff'})
fi_df1

,feature,corr_coeff
0,sector,-0.202913
1,price,1.000000
2,price_per_sqft,0.793641
3,bedRoom,0.572662
4,bathroom,0.610240
5,balcony,0.274313
6,facing,0.021542
7,built_up_area,0.724482
8,servant room,0.395196
9,age_category,-0.125362


In [56]:
from sklearn.ensemble import RandomForestRegressor

# Train a Random Forest regressor on label encoded data
rf_label = RandomForestRegressor(n_estimators=100, random_state=42)
rf_label.fit(X_label, y_label)

# Extract feature importance scores for label encoded data
fi_df2 = pd.DataFrame({
    'feature': X_label.columns,
    'rf_importance': rf_label.feature_importances_
}).sort_values(by='rf_importance', ascending=False)

fi_df2

,feature,rf_importance
2,price_per_sqft,0.594594
7,built_up_area,0.372686
12,dist_cyber_city_km,0.006529
1,sector,0.005667
3,bedRoom,0.003948
6,facing,0.003617
4,bathroom,0.003120
13,luxury_category,0.002645
9,age_category,0.001794
5,balcony,0.001297


In [57]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_train_label, X_test_label, y_train_label, y_test_label = train_test_split(X_label, y_label, test_size=0.2, random_state=42)

# Train a Random Forest regressor on label encoded data
rf_label = RandomForestRegressor(n_estimators=100, random_state=42)
rf_label.fit(X_train_label, y_train_label)

# Calculate Permutation Importance
perm_importance = permutation_importance(rf_label, X_test_label, y_test_label, n_repeats=30, random_state=42)

# Organize results into a DataFrame
fi_df3 = pd.DataFrame({
    'feature': X_label.columns,
    'permutation_importance': perm_importance.importances_mean
}).sort_values(by='permutation_importance', ascending=False)

fi_df3

,feature,permutation_importance
2,price_per_sqft,0.708273
7,built_up_area,0.702019
3,bedRoom,0.004489
12,dist_cyber_city_km,0.003162
1,sector,0.001704
5,balcony,0.000776
6,facing,0.000564
8,servant room,0.000434
0,property_type,0.000394
14,floor_category,0.000244


In [58]:
# although not very reliable because i have applied ordinalencoder for applying linear models need to apply one hot encoding
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_label)

# Train a LASSO regression model
# We'll use a relatively small value for alpha (the regularization strength) for demonstration purposes
lasso = Lasso(alpha=0.01, random_state=42)
lasso.fit(X_scaled, y_label)

# Extract coefficients
fi_df4 = pd.DataFrame({
    'feature': X_label.columns,
    'lasso_coeff': lasso.coef_
}).sort_values(by='lasso_coeff', ascending=False)

fi_df4

,feature,lasso_coeff
2,price_per_sqft,1.731925
7,built_up_area,1.401034
3,bedRoom,0.092491
8,servant room,0.079948
1,sector,0.052508
13,luxury_category,0.045765
4,bathroom,0.018937
10,furnishing_type,-0.000000
12,dist_cyber_city_km,-0.000000
14,floor_category,0.000000


In [59]:
from sklearn.feature_selection import RFE

# Initialize the base estimator
estimator = RandomForestRegressor()

# Apply RFE on the label-encoded and standardized training data
selector_label = RFE(estimator, n_features_to_select=X_label.shape[1], step=1)
selector_label = selector_label.fit(X_label, y_label)

# Get the selected features based on RFE
selected_features = X_label.columns[selector_label.support_]

# Extract the coefficients for the selected features from the underlying linear regression model
selected_coefficients = selector_label.estimator_.feature_importances_

# Organize the results into a DataFrame
fi_df5 = pd.DataFrame({
    'feature': selected_features,
    'rfe_score': selected_coefficients
}).sort_values(by='rfe_score', ascending=False)

fi_df5

,feature,rfe_score
2,price_per_sqft,0.603573
7,built_up_area,0.364762
12,dist_cyber_city_km,0.007014
1,sector,0.004349
4,bathroom,0.003466
3,bedRoom,0.003464
6,facing,0.003310
13,luxury_category,0.002947
5,balcony,0.001370
9,age_category,0.001300


In [60]:
# Train a linear regression model on the label-encoded and standardized training data
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X_scaled, y_label)

# Extract coefficients
fi_df6 = pd.DataFrame({
    'feature': X_label.columns,
    'reg_coeffs': lin_reg.coef_
}).sort_values(by='reg_coeffs', ascending=False)

fi_df6

,feature,reg_coeffs
2,price_per_sqft,1.754758
7,built_up_area,1.412375
3,bedRoom,0.106255
8,servant room,0.090541
1,sector,0.068627
13,luxury_category,0.054640
4,bathroom,0.026752
14,floor_category,0.004788
9,age_category,0.002246
10,furnishing_type,0.001335


In [61]:
final_fi_df = fi_df1.merge(fi_df2,on='feature').merge(fi_df3,on='feature').merge(fi_df4,on='feature').merge(fi_df5,on='feature').merge(fi_df6,on='feature').set_index('feature')

In [62]:
final_fi_df

,corr_coeff,rf_importance,permutation_importance,lasso_coeff,rfe_score,reg_coeffs
feature,,,,,,
sector,-0.202913,0.005667,0.001704,0.052508,0.004349,0.068627
price_per_sqft,0.793641,0.594594,0.708273,1.731925,0.603573,1.754758
bedRoom,0.572662,0.003948,0.004489,0.092491,0.003464,0.106255
bathroom,0.610240,0.003120,-0.004790,0.018937,0.003466,0.026752
balcony,0.274313,0.001297,0.000776,-0.093846,0.001370,-0.123236
facing,0.021542,0.003617,0.000564,-0.021686,0.003310,-0.033956
built_up_area,0.724482,0.372686,0.702019,1.401034,0.364762,1.412375
servant room,0.395196,0.000889,0.000434,0.079948,0.000648,0.090541
age_category,-0.125362,0.001794,0.000098,0.000000,0.001300,0.002246


In [63]:
# normalize the score
final_fi_df = final_fi_df.divide(final_fi_df.sum(axis=0), axis=1)

In [66]:
final_fi_df[['rf_importance','permutation_importance','rfe_score']].mean(axis=1).sort_values(ascending=False)

feature
price_per_sqft        0.566310
built_up_area         0.411190
dist_cyber_city_km    0.005261
sector                0.003741
bedRoom               0.003529
facing                0.002443
luxury_category       0.001610
balcony               0.001072
bathroom              0.001069
age_category          0.001055
combined_rating       0.000731
furnishing_type       0.000702
floor_category        0.000672
servant room          0.000615
dtype: float64